# Self-Attention & Scaled Dot-Product — AI Lab Instructor Notebook

*NLP & Transformers · Hard*



7 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)


## Task 1: Dataset & Setup
*Section: Setup*

# Self-Attention & Scaled Dot-Product — Student Lab

Implement the core attention primitive in NumPy:
- scaled dot-product attention
- masking (padding + causal)
- multi-head attention (forward only)

Goal: correct shapes + numerical stability + intuition.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Self-Attention & Scaled Dot-Product — Instructor Lab

Complete forward-pass implementations + short rationales. Offline and deterministic.

## Stable softmax
Rationale: attention weights are a softmax over scores; stability matters because scores can be large.

## Task 2: Section 0 — Stable softmax
*Section: Section 0*

## Section 0 — Stable softmax

### Task 0.1
Implement `softmax(x, axis)` that is stable by subtracting the axis-wise maximum first.

In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    m = np.max(x, axis=axis, keepdims=True)
    z = x - m
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)

x = rng.standard_normal((3, 5)) * 50
p = softmax(x, axis=1)
check('rowsum1', float(np.max(np.abs(p.sum(axis=1) - 1.0))) < 1e-8)

In [ ]:
# Checks
check('rowsum1', float(np.max(np.abs(p.sum(axis=1) - 1.0))) < 1e-8)

## Task 3: Section 1 — Scaled dot-product attention
*Section: Section 1*

## Section 1 — Scaled dot-product attention

We use shapes:
- `Q`: (B, Tq, d)
- `K`: (B, Tk, d)
- `V`: (B, Tk, dv)

Attention scores: `S = Q @ K^T / sqrt(d)` with `S`: (B, Tq, Tk).
Weights: `A = softmax(S)` over last axis.
Output: `O = A @ V` => (B, Tq, dv).

### Task 1.1
Implement `scaled_dot_attn(Q, K, V, mask=None)` returning `(O, A)`.
If `mask` is provided, treat it as **1 = keep, 0 = mask** and make it broadcastable to `(B, Tq, Tk)`.

In [ ]:
def scaled_dot_attn(Q: np.ndarray, K: np.ndarray, V: np.ndarray, mask = None):
    d = Q.shape[-1]
    scores = (Q @ np.swapaxes(K, -1, -2)) / np.sqrt(d)
    if mask is not None:
        scores = np.where(mask.astype(bool), scores, -1e9)
    A = softmax(scores, axis=-1)
    O = A @ V
    return O, A

B, T, D = 2, 4, 8
Q = rng.standard_normal((B, T, D))
K = rng.standard_normal((B, T, D))
V = rng.standard_normal((B, T, D))
O, A = scaled_dot_attn(Q, K, V)
check('shapes', O.shape == (B, T, D) and A.shape == (B, T, T))
check('rowsum', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

In [ ]:
# Checks
check('shapes', O.shape == (B, T, D) and A.shape == (B, T, T))
check('rowsum', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

**Why this works.** ## Scaled dot-product attention
Rationale: scale by `1/sqrt(d)` to keep score variance controlled, preventing softmax saturation.

## Task 4: Section 2.1 — Masks
*Section: Section 2*

## Section 2 — Masks

### Task 2.1
Implement:
- `causal_mask(T)` returning shape `(1, T, T)` with 1 on/below the diagonal
- `padding_mask(lengths, T)` returning shape `(B, 1, T)` with 1 for valid tokens and 0 for padded positions

In [ ]:
def causal_mask(T: int) -> np.ndarray:
    return np.tril(np.ones((T, T), dtype=np.int64))[None, :, :]

def padding_mask(lengths: np.ndarray, T: int) -> np.ndarray:
    idx = np.arange(T)[None, :]
    m = (idx < lengths[:, None]).astype(np.int64)
    return m[:, None, :]

D = 8
T = 5
Q = rng.standard_normal((1, T, D))
K = rng.standard_normal((1, T, D))
V = rng.standard_normal((1, T, D))
O1, A1 = scaled_dot_attn(Q, K, V, mask=causal_mask(T))
check('causal_zero_future', float(A1[0, 0, 1:].sum()) < 1e-8)

In [ ]:
# Checks
check('causal_zero_future', float(A1[0, 0, 1:].sum()) < 1e-8)

**Why this works.** ## Masks
Rationale: masking must happen before softmax; causal masks prevent attending to future positions, and padding masks hide invalid keys.

## Task 5: Section 2.2 — Masking effects
*Section: Section 2*

### Task 2.2
Show masking effects by comparing attention weights with and without masks.
Use both a causal mask and a padding mask and verify the masked weights become zero where they should.

In [ ]:
D = 8
T = 5
Q = rng.standard_normal((1, T, D))
K = rng.standard_normal((1, T, D))
V = rng.standard_normal((1, T, D))

O0, A0 = scaled_dot_attn(Q, K, V)
O1, A1 = scaled_dot_attn(Q, K, V, mask=causal_mask(T))
lengths = np.array([3], dtype=np.int64)
O2, A2 = scaled_dot_attn(Q, K, V, mask=padding_mask(lengths, T))

print('no_mask first row', np.round(A0[0, 0], 3))
print('causal first row', np.round(A1[0, 0], 3))
print('padding first row', np.round(A2[0, 0], 3))
check('causal_zero_future', float(A1[0, 0, 1:].sum()) < 1e-8)
check('padding_zero_padded_cols', float(A2[0, 0, 3:].sum()) < 1e-8)

In [ ]:
# Checks
check('causal_zero_future', float(A1[0, 0, 1:].sum()) < 1e-8)
check('padding_zero_padded_cols', float(A2[0, 0, 3:].sum()) < 1e-8)

**Why this works.** ## Masking effects
Rationale: visualizing weight rows makes it obvious that causal masks remove future positions and padding masks remove invalid keys.

## Task 6: Section 3 — Self-attention layer (single-head)
*Section: Section 3*

## Section 3 — Self-attention layer (single-head)

Given input `X` with shape `(B, T, Dmodel)`: 
- `Q = X Wq`
- `K = X Wk`
- `V = X Wv`
- `O = Attn(Q, K, V)`
- `Y = O Wo`

### Task 3.1
Implement `self_attn_forward(X, params, mask=None)` returning `(Y, A)`.

In [ ]:
def self_attn_forward(X: np.ndarray, params: dict, mask = None):
    Wq, Wk, Wv, Wo = params['Wq'], params['Wk'], params['Wv'], params['Wo']
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    O, A = scaled_dot_attn(Q, K, V, mask=mask)
    Y = O @ Wo
    return Y, A

B, T, Dm = 2, 6, 12
X = rng.standard_normal((B, T, Dm))
params = {
  'Wq': 0.02 * rng.standard_normal((Dm, Dm)),
  'Wk': 0.02 * rng.standard_normal((Dm, Dm)),
  'Wv': 0.02 * rng.standard_normal((Dm, Dm)),
  'Wo': 0.02 * rng.standard_normal((Dm, Dm)),
}
Y, A = self_attn_forward(X, params, mask=causal_mask(T))
check('shapes', Y.shape == (B, T, Dm) and A.shape == (B, T, T))
check('rowsum', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

In [ ]:
# Checks
check('shapes', Y.shape == (B, T, Dm) and A.shape == (B, T, T))
check('rowsum', float(np.max(np.abs(A.sum(axis=-1) - 1.0))) < 1e-8)

**Why this works.** ## Self-attention layer (single-head)
Rationale: self-attention is attention where Q, K, and V are all learned projections of the same input sequence.

## Task 7: Section 4 — Multi-head attention (forward only)
*Section: Section 4*

## Section 4 — Multi-head attention (forward only)

### Task 4.1
Implement `mha_forward(X, params, n_heads, mask=None)` with shapes:
- input `X`: `(B, T, Dm)`
- split into heads: `(B, h, T, Dh)` where `Dh = Dm / h`
Return `Y`: `(B, T, Dm)` and `A`: `(B, h, T, T)`.

In [ ]:
# No source solution — left as an exercise.